# Biohub - Cell Tracking S2.00 3D U-Net Training Notebook

このノートブックは、**3D U-Net による 3D Center Heatmap 回帰モデル** を Kaggle 環境で学習し、最適重みファイル (`3dunet_center_heatmap_best.pth`) を生成するための学習専用ノートブックです。

## 概要と学習目的

顕微鏡3D画像における暗い細胞や密集した細胞の検出漏れ(False Negative: FN)を最小化するため、GT(正解)細胞重心座標にガウシアン球を付与した **3D Center Heatmap** を正解ターゲットとして、**Gaussian Focal Loss** を用いて 3D U-Net を高精度に学習します。

学習完了後に生成される `3dunet_center_heatmap_best.pth` を Kaggle Dataset (`bct-3dunet-weights`) に登録し、推論ノートブック (`s2_01_Detection_3DUNet+CenterHeatmap.ipynb`) にマウントして提出用予測を実行します。

## Kaggle 本番環境のディレクトリ構成

```text
/kaggle/
├── working/                                         # 作業ディレクトリ (重み出力先)
│   ├── s2_00_train_3dunet.ipynb                    # 実行学習ノートブック
│   └── 3dunet_center_heatmap_best.pth              # 出力される最適重みファイル
└── input/
    ├── competitions/
    │   └── biohub-cell-tracking-during-development/ # コンペ公式データセット
    │       └── train/                              # 訓練用データセット (.zarr / .geff)
    └── datasets/
        └── aaaa1597/
            └── zarr-offline-installation-wheels/  # zarr オフラインインストール用Wheels
                └── zarr_wheels/
```


## 処理フローチャート (Training Pipeline Flowchart)

```mermaid
graph TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef loop fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    classDef cell fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef func fill:#efebe9,stroke:#5d4037,stroke-width:1px;
    classDef cond fill:#fff9c4,stroke:#fbc02d,stroke-width:1px;

    Start([学習処理開始]) --> Cell3["Cell 3: ハイパーパラメータ設定<br>(Epochs, BatchSize, LR, PatchSize)"]
    Cell3 --> Cell4["Cell 4: check_enviroment() <br>(zarr導入 & GPU確認)"]
    class Cell4 func;
    
    Cell4 --> LoadData["Cell 5: Zarr & GT座標データのロード<br>+ 3D Patch Dataset & DataLoader構築"]
    class LoadData func;
    
    LoadData --> BuildModel["Cell 6: 3D U-Net & Gaussian Focal Loss の定義"]
    
    BuildModel --> TrainLoop["Cell 7: 学習ループの開始<br>(AdamW + CosineAnnealingLR)"]
    
    subgraph LoopGroup ["Epoch ループ (1..N Epochs)"]
        EpochStart{"Epoch 開始"}
        class EpochStart loop;
        
        EpochStart --> TrainStep["1. 3D パッチの順伝播 + Focal Loss 計算"]
        TrainStep --> Backprop["2. 逆伝播 (Optimizer Step)"]
        Backprop --> ValStep["3. 検証(Validation) Loss 計算"]
        
        ValStep --> CheckBest{"Validation Loss が過去最小?"}
        class CheckBest cond;
        
        CheckBest -- Yes --> SaveBest["3dunet_center_heatmap_best.pth 保存"]
        CheckBest -- No --> EpochEnd[ ]
        SaveBest --> EpochEnd
        style EpochEnd fill:none,stroke:none,width:0px,height:0px;
    end

    TrainLoop --> EpochStart
    EpochEnd --> LoopNext{"次の Epoch あり?"}
    class LoopNext loop;
    LoopNext -- Yes --> EpochStart
    LoopNext -- No --> End([学習完了: 重み出力])
```


In [ ]:
# === Cell 3: ハイパーパラメータ設定 & パス設定 ===
import datetime
print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 3: パラメータ設定 開始")

import os
import sys

# ==========================================
#   TRAINING HYPERPARAMETERS(学習パラメータ)
# ==========================================

# 1. エポック数 & バッチサイズ
EPOCHS = 10
BATCH_SIZE = 4
LEARNING_RATE = 1e-3

# 2. 3D パッチサイズ (Z, Y, X)
PATCH_SIZE = (32, 64, 64)

# 3. GT 3D ガウシアンヒートマップ設定 (異方性シグマ)
SIGMA_XY = 1.5   # xy方向のガウシアン広がり [voxel]
SIGMA_Z  = 1.0   # z方向のガウシアン広がり [voxel]

# 4. 1データセットから抽出する 3D パッチ数
PATCHES_PER_DATASET = 40

# 5. 入力データパス & 重み出力先
DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
OUTPUT_WEIGHTS_PATH = "3dunet_center_heatmap_best.pth"

print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 3: パラメータ設定 完了")
print(f"  PATCH_SIZE: {PATCH_SIZE}, EPOCHS: {EPOCHS}, BATCH_SIZE: {BATCH_SIZE}, LR: {LEARNING_RATE}")


In [ ]:
# === Cell 4: check_enviroment() (環境確認 & オフラインzarrインストール) ===
import datetime
print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 4: check_enviroment() 開始")

import subprocess
import torch

def check_enviroment():
    """
    Kaggle実行環境の自動検証・zarr オフラインインポート確認
    """
    print("--- 1. Python & PyTorch / CUDA 環境チェック ---")
    print(f"Python sys.version: {sys.version}")
    print(f"PyTorch Version: {torch.__version__}")
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available: {cuda_available}")
    if cuda_available:
        print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    
    print("\n--- 2. オフライン zarr インストールチェック ---")
    try:
        import zarr
        print(f"zarr already installed (version: {zarr.__version__})")
    except ImportError:
        print("zarr not found. Installing zarr from offline wheels...")
        cmd = "!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr"
        print(f"Executing: {cmd}")
        subprocess.run(["pip", "install", "--no-index", "--find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels", "zarr"], check=False)

    os.makedirs("outputs", exist_ok=True)
    print("\n>>> check_enviroment(): SUCCESS! 環境初期化が完了しました。")

check_enviroment()


In [ ]:
# === Cell 5: 3D パッチデータセット & 3D GT Center Heatmap 生成モジュール ===
import numpy as np
import pandas as pd
import zarr
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import gaussian_filter

def generate_gaussian_heatmap_3d(shape, centroids_voxel, sigma_z=1.0, sigma_xy=1.5):
    """
    指定された 3D 形状 (Z, Y, X) 内の細胞重心に 3D ガウシアン球を重畳した Center Heatmap を生成
    """
    heatmap = np.zeros(shape, dtype=np.float32)
    if len(centroids_voxel) == 0:
        return heatmap

    for cz, cy, cx in centroids_voxel:
        iz, iy, ix = int(round(cz)), int(round(cy)), int(round(cx))
        if 0 <= iz < shape[0] and 0 <= iy < shape[1] and 0 <= ix < shape[2]:
            heatmap[iz, iy, ix] = 1.0

    # 異方性ガウシアンフィルタによる滑らかな球体の生成
    heatmap = gaussian_filter(heatmap, sigma=(sigma_z, sigma_xy, sigma_xy))
    h_max = heatmap.max()
    if h_max > 0:
        heatmap = heatmap / h_max # 0.0 ~ 1.0 正規化
    return heatmap

class CellPatchDataset3D(Dataset):
    """
    3D 画像から細胞重心周辺の 3D パッチをランダムサンプリングする Dataset
    """
    def __init__(self, data_dir, patch_size=(32, 64, 64), patches_per_dataset=40, sigma_z=1.0, sigma_xy=1.5):
        self.patch_size = patch_size
        self.patches_per_dataset = patches_per_dataset
        self.sigma_z = sigma_z
        self.sigma_xy = sigma_xy
        
        self.zarr_list = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if d.endswith('.zarr')] if os.path.exists(data_dir) else []
        self.items = []
        
        print(f"Building 3D Patch Dataset from {len(self.zarr_list)} Zarr videos...")
        for z_path in self.zarr_list:
            try:
                root = zarr.open(z_path, mode='r')
                img_data = root['data'] if 'data' in root else root[list(root.keys())[0]]
                attrs = dict(root.attrs)
                voxel_spacing = attrs.get('voxel_spacing', [1.0, 1.0, 1.0])
                
                # トラック情報・GT座標の取得 (tracks.csv または attrs)
                tracks_csv = os.path.join(z_path, 'tracks.csv')
                if os.path.exists(tracks_csv):
                    df_gt = pd.read_csv(tracks_csv)
                else:
                    df_gt = pd.DataFrame()
                    
                self.items.append({
                    'zarr_path': z_path,
                    'shape': img_data.shape,
                    'voxel_spacing': voxel_spacing,
                    'df_gt': df_gt
                })
            except Exception as e:
                print(f"Error indexing {z_path}: {e}")

    def __len__(self):
        return len(self.items) * self.patches_per_dataset

    def __getitem__(self, idx):
        item_idx = idx % len(self.items)
        item = self.items[item_idx]
        
        z_path = item['zarr_path']
        root = zarr.open(z_path, mode='r')
        img_arr = root['data'] if 'data' in root else root[list(root.keys())[0]]
        
        num_frames, depth, height, width = img_arr.shape
        pz, py, px = self.patch_size
        
        # ランダムなフレームとパッチ位置の決定
        f_idx = np.random.randint(0, num_frames)
        vol_3d = np.array(img_arr[f_idx], dtype=np.float32)
        
        sz = np.random.randint(0, max(1, depth - pz))
        sy = np.random.randint(0, max(1, height - py))
        sx = np.random.randint(0, max(1, width - px))
        
        crop_img = vol_3d[sz:sz+pz, sy:sy+py, sx:sx+px]
        
        # パッチ形状を正確にリサイズ/パディング
        cz, cy, cx = crop_img.shape
        if (cz, cy, cx) != self.patch_size:
            pad_img = np.zeros(self.patch_size, dtype=np.float32)
            pad_img[:cz, :cy, :cx] = crop_img
            crop_img = pad_img

        # 正規化 (0.0 ~ 1.0)
        c_min, c_max = crop_img.min(), crop_img.max()
        if c_max > c_min:
            crop_img = (crop_img - c_min) / (c_max - c_min)
            
        # GT 3D Center Heatmap の生成
        df_gt = item['df_gt']
        centroids_voxel = []
        if len(df_gt) > 0 and 't' in df_gt.columns:
            frame_gt = df_gt[df_gt['t'] == f_idx]
            z_scale, y_scale, x_scale = item['voxel_spacing']
            for row in frame_gt.itertuples():
                gz_v = row.z / z_scale - sz
                gy_v = row.y / y_scale - sy
                gx_v = row.x / x_scale - sx
                if 0 <= gz_v < pz and 0 <= gy_v < py and 0 <= gx_v < px:
                    centroids_voxel.append((gz_v, gy_v, gx_v))
                    
        gt_heatmap = generate_gaussian_heatmap_3d(
            self.patch_size,
            centroids_voxel,
            sigma_z=self.sigma_z,
            sigma_xy=self.sigma_xy
        )

        tensor_img = torch.from_numpy(crop_img).unsqueeze(0)        # Shape: (1, Z, Y, X)
        tensor_target = torch.from_numpy(gt_heatmap).unsqueeze(0)   # Shape: (1, Z, Y, X)
        return tensor_img, tensor_target

print("Cell 5: 3D パッチデータセット & 3D GT Center Heatmap 生成モジュール定義完了")


In [ ]:
# === Cell 6: 3D U-Net アーキテクチャ & Gaussian Focal Loss の定義 ===
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet3D(nn.Module):
    """
    3D U-Net Architecture for Center Heatmap Regression
    """
    def __init__(self, in_channels=1, out_channels=1, base_filters=16):
        super().__init__()
        f = base_filters
        self.enc1 = ConvBlock3D(in_channels, f)
        self.pool1 = nn.MaxPool3d(2)
        
        self.enc2 = ConvBlock3D(f, f*2)
        self.pool2 = nn.MaxPool3d(2)
        
        self.bottleneck = ConvBlock3D(f*2, f*4)
        
        self.up2 = nn.ConvTranspose3d(f*4, f*2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock3D(f*4, f*2)
        
        self.up1 = nn.ConvTranspose3d(f*2, f, kernel_size=2, stride=2)
        self.dec1 = ConvBlock3D(f*2, f)
        
        self.final_conv = nn.Conv3d(f, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        b = self.bottleneck(p2)
        
        u2 = self.up2(b)
        if u2.shape != e2.shape:
            u2 = F.interpolate(u2, size=e2.shape[2:], mode='trilinear', align_corners=True)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        if u1.shape != e1.shape:
            u1 = F.interpolate(u1, size=e1.shape[2:], mode='trilinear', align_corners=True)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        out = torch.sigmoid(self.final_conv(d1))
        return out

class GaussianFocalLoss(nn.Module):
    """
    CenterNet Style Gaussian Focal Loss for 3D Heatmap Regression
    """
    def __init__(self, alpha=2.0, beta=4.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, pred, target):
        pos_mask = target.eq(1.0)
        neg_mask = target.lt(1.0)

        neg_weights = torch.pow(1.0 - target, self.beta)

        pos_loss = torch.log(pred + 1e-6) * torch.pow(1.0 - pred, self.alpha) * pos_mask
        neg_loss = torch.log(1.0 - pred + 1e-6) * torch.pow(pred, self.alpha) * neg_weights * neg_mask

        num_pos = pos_mask.float().sum()
        pos_loss = pos_loss.sum()
        neg_loss = neg_loss.sum()

        if num_pos == 0:
            loss = -neg_loss
        else:
            loss = -(pos_loss + neg_loss) / num_pos
        return loss

print("Cell 6: 3D U-Net アーキテクチャ & Gaussian Focal Loss 定義完了")


In [ ]:
# === Cell 7: 3D U-Net 学習ループ & 最適重み保存 ===
import datetime
print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> Cell 7: 学習ループ 開始")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Executing Training on Device: {device}")

# 1. Dataset & DataLoader の準備
dataset = CellPatchDataset3D(
    DATA_DIR,
    patch_size=PATCH_SIZE,
    patches_per_dataset=PATCHES_PER_DATASET,
    sigma_z=SIGMA_Z,
    sigma_xy=SIGMA_XY
)

if len(dataset) == 0:
    print("Warning: Dataset is empty. Check DATA_DIR path.")
    dataloader = []
else:
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# 2. モデル, Loss, Optimizer, Scheduler の初期化
model = UNet3D(in_channels=1, out_channels=1, base_filters=16).to(device)
criterion = GaussianFocalLoss(alpha=2.0, beta=4.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')

# 3. 学習ループ
if len(dataloader) > 0:
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        batch_count = 0
        
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            batch_count += 1
            
        scheduler.step()
        epoch_loss = running_loss / max(1, batch_count)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%H:%M:%S')}] Epoch [{epoch:02d}/{EPOCHS:02d}] - Loss: {epoch_loss:.6f} | LR: {current_lr:.6f}")
        
        # 最適重みの更新と保存
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), OUTPUT_WEIGHTS_PATH)
            print(f"  >>> Best model weights updated and saved to: {OUTPUT_WEIGHTS_PATH} (Best Loss: {best_loss:.6f})")

print(f"\n[{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')} JST] >>> 学習処理完了! 最適重みファイル: {OUTPUT_WEIGHTS_PATH}")
